In [47]:
import tmu
from PyHierarchicalTsetlinMachineCUDA.tm import TsetlinMachine
import PyHierarchicalTsetlinMachineCUDA.tm as hier_tm_objects
from scipy.special import expit

import numpy as np
import pandas as pd
from chembl_structure_pipeline.standardizer import standardize_mol as chembl_standardizer
from rdkit import Chem
from rdkit.Chem import DataStructs, rdFingerprintGenerator

from kennard_stone import train_test_split as ks_train_test_split
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    auc,
    mean_absolute_error,
    precision_recall_curve,
    precision_score,
    roc_auc_score,
    roc_curve,
    root_mean_squared_error,
)


In [2]:

def mol_from_smiles(smiles, *args, standardizer=chembl_standardizer) -> Chem.Mol:
    """Convert a SMILES string to an RDKit Mol object.

    :param smiles: SMILES string. Required.
    :param standardizer: function to standardize the molecule. Defaults to
        ChEMBL standardizer. Use None to skip.
    :return: RDKit Mol object, or None if the SMILES could not be parsed.
    """
    mol = Chem.MolFromSmiles(str(smiles))
    if not mol:
        return None
    if standardizer is None:
        return mol
    return standardizer(mol, check_exclusion=True, sanitize=True)


def fp_to_np(fp):
    arr = np.zeros((1,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def gen_ecfp_arr(mol_df, mol_col, fp_size=1024, fp_radius=2, n_threads=-1):
    """Morgan (ECFP)-style fingerprints as a binary uint32 array."""
    mols = list(mol_df[mol_col])
    fpg = rdFingerprintGenerator.GetMorganGenerator(radius=fp_radius, fpSize=fp_size)
    fps = fpg.GetFingerprints(mols, numThreads=n_threads)
    fps_np = np.array([fp_to_np(i) for i in fps], dtype=np.uint32)
    return fps_np

In [3]:
def prc_auc_score(Y, Y_pred):
    """Area under the precision-recall curve."""
    precision, recall, _ = precision_recall_curve(Y, Y_pred)
    return auc(recall, precision)

def ppv_npv_score(Y, Y_pred):
    """Positive/negative predictive value at the ROC-optimal (Youden's J)
    probability cutoff."""
    fpr, tpr, proba = roc_curve(Y, Y_pred)
    optimal_proba_cutoff = sorted(
        list(zip(np.abs(tpr - fpr), proba)), key=lambda i: i[0], reverse=True
    )[0][1]

    hard_Y_pred = [1 if p > optimal_proba_cutoff else 0 for p in Y_pred]

    return (
        precision_score(Y, hard_Y_pred, pos_label=1),
        precision_score(Y, hard_Y_pred, pos_label=0),
    )

In [4]:
mor_data = pd.read_csv("data/MOR.csv")
# Calculate descriptors for the MOR dataset
mor_data["mol"] = mor_data["SMILES"].apply(lambda x: Chem.MolFromSmiles(x))
X = gen_ecfp_arr(mor_data, "mol")


In [48]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, mor_data["label"], test_size=0.2)

# equal amounts train and test sets
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2)

In [ ]:
![Description of the image](path/to/image.png)

In [ ]:
# one shot

# before --> too many clauses
#        --> s too low
#        --> T too high


n_clauses = 20
T = 100000
s = 10
weighted_clauses = True
n_concepts = 5

tm = TsetlinMachine(
    number_of_clauses=n_clauses,
    T=T,
    s=s,
    weighted_clauses=weighted_clauses,
    hierarchy_structure = (
        (hier_tm_objects.AND_GROUP, 1024),
        (hier_tm_objects.OR_ALTERNATIVES, 16),
        (hier_tm_objects.AND_ALTERNATIVES, n_concepts)
    )
)


In [50]:
for epoch in range(50):
    tm.fit(X_train_val, y_train_val)
    
    result = 100*(tm.predict(X_test) == y_test).mean()
    predict_probabilities = expit(tm.score(X_test)/tm.T)
    roc_auc = roc_auc_score(y_test, predict_probabilities)
    prc = prc_auc_score(y_test, predict_probabilities)
    ppv,npv = ppv_npv_score(y_test, predict_probabilities)
    print(f"Epoch {epoch+1}: Test Accuracy = {result:.2f}%, ROC AUC = {roc_auc:.2f}, PRC AUC = {prc:.2f}, PPV = {ppv:.2f}, NPV = {npv:.2f}")

Epoch 1: Test Accuracy = 77.51%, ROC AUC = 0.88, PRC AUC = 0.80, PPV = 0.67, NPV = 0.91
Epoch 2: Test Accuracy = 83.45%, ROC AUC = 0.91, PRC AUC = 0.86, PPV = 0.68, NPV = 0.92
Epoch 3: Test Accuracy = 83.45%, ROC AUC = 0.92, PRC AUC = 0.86, PPV = 0.69, NPV = 0.94
Epoch 4: Test Accuracy = 81.47%, ROC AUC = 0.90, PRC AUC = 0.85, PPV = 0.64, NPV = 0.94
Epoch 5: Test Accuracy = 83.17%, ROC AUC = 0.89, PRC AUC = 0.85, PPV = 0.74, NPV = 0.91
Epoch 6: Test Accuracy = 85.86%, ROC AUC = 0.92, PRC AUC = 0.89, PPV = 0.72, NPV = 0.94
Epoch 7: Test Accuracy = 87.13%, ROC AUC = 0.93, PRC AUC = 0.88, PPV = 0.75, NPV = 0.93
Epoch 8: Test Accuracy = 85.86%, ROC AUC = 0.92, PRC AUC = 0.88, PPV = 0.71, NPV = 0.94
Epoch 9: Test Accuracy = 86.99%, ROC AUC = 0.92, PRC AUC = 0.86, PPV = 0.75, NPV = 0.92
Epoch 10: Test Accuracy = 86.70%, ROC AUC = 0.92, PRC AUC = 0.89, PPV = 0.70, NPV = 0.94
Epoch 11: Test Accuracy = 83.88%, ROC AUC = 0.89, PRC AUC = 0.85, PPV = 0.65, NPV = 0.93
Epoch 12: Test Accuracy = 87.1

In [ ]:
# given the number of features, I want to divide that number in a pyramid fashion for example if I input 3 and the number of features is 1024, I want to get something like [1024, 32, 1] but the total has to be equal to 1024
# example: input 3 and 1024 --> output [1024, 32, 1]
